## Step 1 imports

In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

## Step 2 load dataset

In [3]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=10000)

C:\Users\SivaGuduri\Desktop\learn-ML\dl-env\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


## Check sample

In [4]:
print(X_train[0])
print(y_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
1


## Step 3 Why padding?

Check different review lengths:

In [5]:
print(len(X_train[0]))
print(len(X_train[1]))
print(len(X_train[2]))

218
189
141


## Step 4 Pad sequences
25k reviews
200 tokens each
This is exactly like resizing images for CNN

In [6]:
X_train = pad_sequences(X_train, maxlen=200)
X_test = pad_sequences(X_test, maxlen=200)

In [7]:
print(X_train.shape)
print(X_test.shape)

(25000, 200)
(25000, 200)


## Step 5 Build model

In [8]:
# RNN MODEL
# model = tf.keras.Sequential([
#     tf.keras.Input(shape=(200,)),
#     tf.keras.layers.Embedding(input_dim=10000, output_dim=64),
#     tf.keras.layers.SimpleRNN(64),
#     tf.keras.layers.Dense(1, activation="sigmoid")
# ])

In [9]:
# # Replace SimpleRNN with LSTM
# model = tf.keras.Sequential([
#     tf.keras.Input(shape=(200,)),
#     tf.keras.layers.Embedding(input_dim=10000, output_dim=64),

#     tf.keras.layers.LSTM(64),

#     tf.keras.layers.Dense(1, activation="sigmoid")
# ])

In [10]:
# Replace LSTM with GRU
model = tf.keras.Sequential([
    tf.keras.Input(shape=(200,)),
    tf.keras.layers.Embedding(input_dim=10000, output_dim=64),

    tf.keras.layers.GRU(64),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

In [11]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 200, 64)             │         640,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru (GRU)                            │ (None, 64)                  │          24,960 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 665,025 (2.54 MB)

 Trainable params: 665,025 (2.54 MB)

 Non-trainable params: 0 (0.00 B)

## Step 6 Train

In [13]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 54s 129ms/step - accuracy: 0.7877 - loss: 0.4335 - val_accuracy: 0.8669 - val_loss: 0.3147
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 57s 147ms/step - accuracy: 0.8972 - loss: 0.2599 - val_accuracy: 0.8453 - val_loss: 0.3705
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 58s 148ms/step - accuracy: 0.9295 - loss: 0.1868 - val_accuracy: 0.8711 - val_loss: 0.3345


## Step 7 Evaluate model

In [15]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.8669 - loss: 0.3147
Test Accuracy: 0.866919994354248
